In [68]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, random_split, DataLoader
import numpy as np
# import pandas as pd
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data Preparation

In [69]:
def bitboard_to_tensor(bitboard):
    array = np.zeros((5, 6, 6), dtype=np.float32) 
    car_dict = defaultdict(list)
    for idx, cell in enumerate(bitboard):
        if cell != "o":
            row, col = divmod(idx, 6)
            car_dict[cell].append((row, col))

    for car, coord in car_dict.items():
        if car == "A":
            channel = 0
        else:
            horizontal = coord[0][0] == coord[1][0]
            length = len(coord)
            channel = (1 if length == 2 else 3) + (0 if horizontal else 1)
        for row, col in coord:
            array[channel][row][col] = True

    return torch.tensor(array)

In [70]:
# col_names = ["distance", "bitboard"]
# df = pd.read_csv("rush_nw.txt", sep=" ", usecols=[0, 1], header=None, names = col_names)
# features = torch.stack([bitboard_to_tensor(b) for b in df["bitboard"]])
# labels = torch.tensor(df["distance"])

In [71]:
data = torch.load("rush_nw_tensors.pt")
features, labels = data["features"], data["labels"]

In [ ]:
dataset = TensorDataset(features, labels, torch.arange(len(features)))
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
generator = torch.Generator().manual_seed(0)
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Neural Network Models

In [73]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(180, 64), nn.ReLU(),
                                 nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x):
        return self.net(x)

In [74]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.net = nn.Sequential(nn.Conv2d(in_channels=5, out_channels=16, kernel_size=3, padding="same"), nn.ReLU(),
                                 nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding="same"), nn.ReLU(),
                                 nn.Flatten(), nn.Linear(1152, 64), nn.ReLU(),
                                 nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x):
        return self.net(x)

# Tests

In [ ]:
def train(model, epochs=10, lr=0.001, momentum=0.9):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        for features, labels, _ in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(features)
            loss = loss_fn(output, labels)
            loss.backward()
            optimizer.step()


def validate(model):
    loss_fn = nn.MSELoss(reduction="sum")
    total = 0.0
    with torch.no_grad():
        for features, labels, _ in test_loader:
            features, labels = features.to(device), labels.to(device)
            predicted = model(features)
            total += loss_fn(predicted, labels).item()
    return total / test_size

In [81]:
torch.manual_seed(42)
mlp_model = MLP().to(device)
train(mlp_model)
mse = validate(mlp_model)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 4.4697626012619285


In [82]:
torch.manual_seed(42)
cnn_model = ConvNet().to(device)
train(cnn_model)
mse = validate(cnn_model)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 17.526843038855567


# Prediction Visualization

In [ ]:
import os
import shutil
import matplotlib.pyplot as plt


def visualize_bitboard(bitboard, title, save_path):
    car_dict = defaultdict(list)
    for idx, cell in enumerate(bitboard):
        if cell != "o":
            row, col = divmod(idx, 6)
            car_dict[cell].append((row, col))

    fig, ax = plt.subplots()
    ax.add_patch(plt.Rectangle((0, 0), 6, 6, facecolor="white", edgecolor="none"))

    for car, coords in car_dict.items():
        rows = [i for i, j in coords]
        cols = [j for i, j in coords]
        i0, i1 = min(rows), max(rows)
        j0, j1 = min(cols), max(cols)
        facecolor = "black" if car == "x" else ("red" if car == "A" else "gray")
        ax.add_patch(plt.Rectangle(
            (j0, i0), j1 - j0 + 1, i1 - i0 + 1,
            facecolor=facecolor,
            edgecolor="black",
            linewidth=1.5,
        ))
        if car not in ("A", "x"):
            ax.text(
                (j0 + j1 + 1) / 2, (i0 + i1 + 1) / 2, car,
                ha="center", va="center", color="white", fontsize=12,
            )

    ax.add_patch(plt.Rectangle(
        (0, 0), 6, 6,
        facecolor="none",
        edgecolor="black",
        linewidth=1.5,
    ))

    ax.set_xlim(0, 6)
    ax.set_ylim(6, 0)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(title, fontsize=10)

    fig.savefig(save_path)
    plt.close(fig)

In [ ]:
def collect_predictions():
    mlp_model.eval()
    cnn_model.eval()
    actuals, mlp_preds, cnn_preds, indices = [], [], [], []
    with torch.no_grad():
        for features, labels, idx in test_loader:
            features = features.to(device)
            actuals.append(labels.squeeze(1))
            mlp_preds.append(mlp_model(features).squeeze(1).cpu())
            cnn_preds.append(cnn_model(features).squeeze(1).cpu())
            indices.append(idx)
    return torch.cat(actuals), torch.cat(mlp_preds), torch.cat(cnn_preds), torch.cat(indices)


actuals, mlp_preds, cnn_preds, indices = collect_predictions()

mlp_error = (mlp_preds - actuals).abs()
cnn_error = (cnn_preds - actuals).abs()
pred_disagreement = (mlp_preds - cnn_preds).abs()
avg_error = (mlp_error + cnn_error) / 2

In [ ]:
n = 10
base_folder = "predictions"

with open("rush_nw.txt") as f:
    lines = f.readlines()

for pos in torch.randperm(len(indices))[:20]:
    row = indices[pos].item()
    assert int(lines[row].split()[0]) == int(actuals[pos].item()), \
        "rush_nw.txt line order doesn't match the dataset row order"

if os.path.exists(base_folder):
    shutil.rmtree(base_folder)
os.makedirs(base_folder)


def save_category(name, score, largest=True):
    positions = torch.topk(score, n, largest=largest).indices
    folder = os.path.join(base_folder, name)
    os.makedirs(folder)
    for rank, pos in enumerate(positions, start=1):
        row = indices[pos].item()
        bitboard = lines[row].split()[1]
        title = f"Actual: {int(actuals[pos].item())}   MLP: {mlp_preds[pos].item():.1f}   CNN: {cnn_preds[pos].item():.1f}"
        save_path = os.path.join(folder, f"puzzle_{rank:03d}.jpg")
        visualize_bitboard(bitboard, title, save_path)
    print(f"Saved {n} frames to {folder}/")


save_category("biggest_disagreement", pred_disagreement)
save_category("mlp_worst", mlp_error)
save_category("cnn_worst", cnn_error)
save_category("avg_worst", avg_error)